# F0 Pipeline Validation (Post Bug-Fix)

Validates the 3 bug fixes + 6/3/3 train/val/test split:
- **BUG-1**: `reg_weights` -> `clf_weights` in classifier branch training
- **BUG-2**: Split scaler dicts into separate clf/reg variants
- **BUG-3**: Threshold optimization now uses 3-month validation set (not in-sample)
- **TrainingConfig**: `train_months=6, val_months=3, test_months=3` (12 months total)

Runs **f0 only** (onpeak + offpeak) across all 21 auction months (2024-06 to 2026-02),
then backtests with `evaluate_signal_v2`.

In [ ]:
from pbase.config.ray import init_ray
import pmodel
import shadow_price_prediction

init_ray(address='ray://10.8.0.36:10001', extra_modules=[pmodel, shadow_price_prediction])

In [ ]:
import gc
import logging
import resource
import shutil
from pathlib import Path

import pandas as pd
from pbase.analysis.tools.all_positions import MisoApTools
from pbase.data.dataset.signal.general import ConstraintsSignal

from shadow_price_prediction import (
    ShadowPricePipeline,
    PredictionConfig,
    FeatureConfig,
    ModelConfig,
    ModelSpec,
    EnsembleConfig,
    TrainingConfig,
    ThresholdConfig,
    AnomalyDetectionConfig,
    XGBClassifier, XGBRegressor,
    LogisticRegression, ElasticNet,
    generate_and_save_signal,
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


def mem_mb():
    """Current process peak RSS in MB."""
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024


print(f"Imports done. Memory: {mem_mb():.0f} MB")

In [ ]:
# === Constants ===
RTO = 'miso'
SIGNAL_NAME = 'TEST.TEST.Signal.MISO.SPICE_F0P_V6.7B.R1'
MARKET_ROUND = 1

# New output dir (v2) so we don't collide with old results
OUTPUT_DIR = '/opt/temp/tmp/pw_data/spice6/signal_67b_results_v2'
OLD_OUTPUT_DIR = '/opt/temp/tmp/pw_data/spice6/signal_67b_results'

# All 21 auction months
AUCTION_MONTHS = pd.date_range('2024-06', '2026-02', freq='MS')

# f0 only: onpeak + offpeak
CLASS_TYPES = ['onpeak', 'offpeak']

print(f"Auction months: {len(AUCTION_MONTHS)} ({AUCTION_MONTHS[0].strftime('%Y-%m')} to {AUCTION_MONTHS[-1].strftime('%Y-%m')})")
print(f"Class types: {CLASS_TYPES}")
print(f"Expected signals: {len(AUCTION_MONTHS) * len(CLASS_TYPES)}")

In [ ]:
# === Feature Configuration ===
# (feature_name, monotonicity_constraint): 1=increasing, -1=decreasing, 0=none

step1_features = [
    ('prob_exceed_110', 1),
    ('prob_exceed_105', 1),
    ('prob_exceed_100', 1),
    ('prob_exceed_95', 1),
    ('prob_exceed_90', 1),
    ('prob_below_95', -1),
    ('prob_below_90', -1),
    ('density_skewness', 1),
]

step2_features = [
    ('prob_exceed_110', 1),
    ('prob_exceed_105', 1),
    ('prob_exceed_100', 1),
    ('prob_exceed_95', 1),
    ('prob_exceed_90', 1),
    ('prob_below_95', -1),
    ('prob_below_90', -1),
    ('density_skewness', 1),
    ('hist_da', 1),
    ('recent_hist_da', 1),
]

feature_config = FeatureConfig(
    step1_features=step1_features,
    step2_features=step2_features,
)

# === Model Configuration (same as generate_signal_67b.ipynb) ===
xgb_clf_spec = ModelSpec(
    model_class=XGBClassifier,
    config=ModelConfig(params={
        'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.1,
        'n_jobs': 1, 'eval_metric': 'logloss',
    }),
    weight=0.5,
)
lr_clf_spec = ModelSpec(
    model_class=LogisticRegression,
    config=ModelConfig(params={
        'C': 1.0, 'max_iter': 1000, 'class_weight': 'balanced', 'n_jobs': 1,
    }),
    weight=0.5,
)
xgb_reg_spec = ModelSpec(
    model_class=XGBRegressor,
    config=ModelConfig(params={
        'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.1, 'n_jobs': 1,
    }),
    weight=0.5,
)
enet_reg_spec = ModelSpec(
    model_class=ElasticNet,
    config=ModelConfig(params={'alpha': 1.0, 'l1_ratio': 0.5}),
    weight=0.5,
)

ensemble_config = EnsembleConfig(
    default_classifiers=[xgb_clf_spec, lr_clf_spec],
    branch_classifiers=[xgb_clf_spec, lr_clf_spec],
    default_regressors=[xgb_reg_spec, enet_reg_spec],
    branch_regressors=[xgb_reg_spec, enet_reg_spec],
)

print(f"Step1: {len(step1_features)} features, Step2: {len(step2_features)} features")
print(f"All features: {feature_config.all_features}")

In [ ]:
# === Archive old f0 results from buggy run (3 auction months) ===
old_dir = Path(OLD_OUTPUT_DIR)
archived_old = {}  # {(auction_month, class_type): DataFrame}

if old_dir.exists():
    for am_dir in sorted(old_dir.glob('auction_month=*')):
        auc_str = am_dir.name.split('=')[1]
        for mm_dir in sorted(am_dir.glob('market_month=*')):
            for ct_dir in sorted(mm_dir.glob('class_type=*')):
                ct = ct_dir.name.split('=')[1]
                fr_path = ct_dir / 'final_results.parquet'
                if fr_path.exists():
                    try:
                        df = pd.read_parquet(fr_path)
                        archived_old[(auc_str, ct)] = df
                        print(f"  Archived: {auc_str}/{ct} — {len(df)} rows")
                    except Exception as e:
                        print(f"  Failed to read {fr_path}: {e}")
    print(f"\nArchived {len(archived_old)} old result sets")
else:
    print(f"No old results directory at {OLD_OUTPUT_DIR}")

In [ ]:
# === Clear old cache ===
if old_dir.exists():
    shutil.rmtree(OLD_OUTPUT_DIR)
    print(f"Removed old cache: {OLD_OUTPUT_DIR}")
else:
    print(f"Old cache already cleared: {OLD_OUTPUT_DIR}")

# Ensure new output dir exists
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"Output dir ready: {OUTPUT_DIR}")

## Phase 1: Smoke Test

Run a single auction month (2025-01) for f0/onpeak with `verbose=True`.
Verify `[DATA SPLIT]` output and "Using validation data" message appear.

In [ ]:
smoke_am = pd.Timestamp('2025-01')
smoke_ct = 'onpeak'

config = PredictionConfig(
    market_round=MARKET_ROUND,
    period_type='f0',
    class_type=smoke_ct,
    features=feature_config,
    models=ensemble_config,
    training=TrainingConfig(
        min_samples_for_branch_model=10,
        train_months=6,
        val_months=3,
        test_months=3,
    ),
    threshold=ThresholdConfig(threshold_beta=2.0),
    anomaly_detection=AnomalyDetectionConfig(enabled=True),
)

pipeline = ShadowPricePipeline(config)

# f0: market_month == auction_month
smoke_results = pipeline.run(
    test_periods=[(smoke_am, smoke_am)],
    class_type=smoke_ct,
    verbose=True,
    use_parallel=False,
    output_dir=OUTPUT_DIR,
)

_, smoke_final, smoke_metrics, _, _ = smoke_results
del pipeline
gc.collect()
print(f"\n[mem] {mem_mb():.0f} MB")

In [ ]:
# === Smoke test assertions ===
assert smoke_final is not None, "smoke_final is None"
assert not smoke_final.empty, "smoke_final is empty"

# Check expected columns exist
expected_cols = {'predicted_shadow_price', 'binding_probability_scaled', 'branch_name'}
actual_cols = set(smoke_final.columns) if not isinstance(smoke_final.index, pd.MultiIndex) else set(smoke_final.reset_index().columns)
missing = expected_cols - actual_cols
assert not missing, f"Missing columns: {missing}"

# Check some constraints are predicted binding
if isinstance(smoke_final.index, pd.MultiIndex):
    n_binding = (smoke_final['predicted_shadow_price'] > 0).sum()
else:
    n_binding = (smoke_final['predicted_shadow_price'] > 0).sum()

print(f"Smoke test PASSED:")
print(f"  Total constraints: {len(smoke_final)}")
print(f"  Binding predictions: {n_binding}")
print(f"  Columns: {sorted(smoke_final.columns.tolist())}")

del smoke_final, smoke_metrics, smoke_results
gc.collect()

## Phase 2: Full f0 Run

Loop 21 auction months x 2 class types = 42 pipeline runs.
For f0, `market_month == auction_month` so `test_periods=[(am, am)]`.

In [ ]:
results_log = []
print(f"Starting full f0 run. Memory: {mem_mb():.0f} MB")
print(f"Runs: {len(AUCTION_MONTHS)} months x {len(CLASS_TYPES)} class types = {len(AUCTION_MONTHS) * len(CLASS_TYPES)}")

for i, auction_month in enumerate(AUCTION_MONTHS):
    auc_str = auction_month.strftime('%Y-%m')

    for class_type in CLASS_TYPES:
        label = f"{auc_str}/f0/{class_type}"
        print(f"\n--- [{i+1}/{len(AUCTION_MONTHS)}] {label} ---")

        config = PredictionConfig(
            market_round=MARKET_ROUND,
            period_type='f0',
            class_type=class_type,
            features=feature_config,
            models=ensemble_config,
            training=TrainingConfig(
                min_samples_for_branch_model=10,
                train_months=6,
                val_months=3,
                test_months=3,
            ),
            threshold=ThresholdConfig(threshold_beta=2.0),
            anomaly_detection=AnomalyDetectionConfig(enabled=True),
        )

        pipeline = ShadowPricePipeline(config)
        final_results = None
        signal_df = None

        try:
            _, final_results, metrics, _, _ = pipeline.run(
                test_periods=[(auction_month, auction_month)],
                class_type=class_type,
                verbose=True,
                use_parallel=False,
                output_dir=OUTPUT_DIR,
            )

            if final_results is not None and not final_results.empty:
                signal_df, save_path = generate_and_save_signal(
                    final_results=final_results,
                    rto=RTO,
                    signal_name=SIGNAL_NAME,
                    period_type='f0',
                    class_type=class_type,
                    auction_month=auction_month,
                )
                n_constraints = len(signal_df) if signal_df is not None and not signal_df.empty else 0
                tier_dist = signal_df['tier'].value_counts().sort_index().to_dict() if n_constraints > 0 else {}
                results_log.append({
                    'auction_month': auc_str, 'class_type': class_type,
                    'status': 'saved', 'n_constraints': n_constraints,
                    'tier_distribution': tier_dist,
                })
                print(f"  => {n_constraints} constraints, tiers: {tier_dist}")
            else:
                results_log.append({
                    'auction_month': auc_str, 'class_type': class_type,
                    'status': 'no_results', 'n_constraints': 0,
                })
                print(f"  => No results")

        except Exception as e:
            logger.error(f"Pipeline failed for {label}: {e}")
            results_log.append({
                'auction_month': auc_str, 'class_type': class_type,
                'status': f'error: {e}', 'n_constraints': 0,
            })
        finally:
            del pipeline, final_results, signal_df
            gc.collect()
            print(f"  [mem] {mem_mb():.0f} MB")

print(f"\nFull run complete. Memory: {mem_mb():.0f} MB")

## Phase 3: Validation

In [ ]:
# === Schema validation: load each signal via ConstraintsSignal ===
EXPECTED_COLUMNS = {
    'constraint_id', 'branch_name', 'flow_direction', 'hist_da',
    'predicted_shadow_price', 'binding_probability_scaled',
    'prob_rank', 'hist_shadow_rank', 'pred_shadow_rank', 'rank', 'tier',
    'shadow_sign', 'shadow_price', 'equipment',
}

validation_errors = []
validated_count = 0

for entry in results_log:
    if entry['status'] != 'saved' or entry['n_constraints'] == 0:
        continue

    am = pd.Timestamp(entry['auction_month'])
    ct = entry['class_type']
    label = f"{entry['auction_month']}/f0/{ct}"

    try:
        sig = ConstraintsSignal(
            rto='miso', signal_name=SIGNAL_NAME,
            period_type='f0', class_type=ct,
        ).load_data(am)

        errors = []
        if set(sig.columns) != EXPECTED_COLUMNS:
            extra = set(sig.columns) - EXPECTED_COLUMNS
            missing = EXPECTED_COLUMNS - set(sig.columns)
            errors.append(f"Columns: extra={extra}, missing={missing}")
        if not sig.index.str.contains('|', regex=False).all():
            errors.append("Index missing '|' separator")
        if not sig.index.str.endswith('|spice').all():
            errors.append("Index not ending with '|spice'")
        if not sig['tier'].isin([0, 1, 2, 3, 4]).all():
            errors.append(f"Invalid tiers: {sig['tier'].unique()}")
        if not (sig['predicted_shadow_price'] > 0).all():
            errors.append("Non-positive predicted_shadow_price")

        if errors:
            validation_errors.extend([(label, e) for e in errors])
            print(f"  FAIL {label}: {errors}")
        else:
            validated_count += 1

    except Exception as e:
        validation_errors.append((label, str(e)))
        print(f"  ERR  {label}: {e}")

if validation_errors:
    print(f"\n{len(validation_errors)} validation errors:")
    for label, err in validation_errors:
        print(f"  {label}: {err}")
else:
    print(f"All {validated_count} signals passed schema validation.")

In [ ]:
# === Pre-fix vs post-fix comparison ===
# Compare archived old results with new results for overlapping months
if archived_old:
    print("Pre-fix vs post-fix comparison (overlapping months):")
    print(f"Old result keys: {list(archived_old.keys())}")
    print()

    for (auc_str, ct), old_df in sorted(archived_old.items()):
        label = f"{auc_str}/f0/{ct}"

        # Load new results from OUTPUT_DIR
        new_path = Path(OUTPUT_DIR) / f'auction_month={auc_str}/market_month={auc_str}/class_type={ct}/final_results.parquet'
        if not new_path.exists():
            print(f"  {label}: new results not found at {new_path}")
            continue

        new_df = pd.read_parquet(new_path)

        # Normalize index for comparison
        if isinstance(old_df.index, pd.MultiIndex):
            old_df = old_df.reset_index()
        if isinstance(new_df.index, pd.MultiIndex):
            new_df = new_df.reset_index()

        # Binding rate
        old_binding = (old_df['predicted_shadow_price'] > 0).mean() if 'predicted_shadow_price' in old_df.columns else float('nan')
        new_binding = (new_df['predicted_shadow_price'] > 0).mean() if 'predicted_shadow_price' in new_df.columns else float('nan')

        # Shadow price correlation for constraints in both
        sp_corr = float('nan')
        if 'constraint_id' in old_df.columns and 'constraint_id' in new_df.columns:
            merged = old_df[['constraint_id', 'predicted_shadow_price']].merge(
                new_df[['constraint_id', 'predicted_shadow_price']],
                on='constraint_id', suffixes=('_old', '_new'),
            )
            if len(merged) > 1:
                sp_corr = merged['predicted_shadow_price_old'].corr(merged['predicted_shadow_price_new'])

        print(f"  {label}:")
        print(f"    Old binding rate: {old_binding:.3f}, New: {new_binding:.3f}")
        print(f"    Shadow price correlation: {sp_corr:.3f}")
        print(f"    Old rows: {len(old_df)}, New rows: {len(new_df)}")
else:
    print("No archived old results to compare against.")

In [ ]:
# === Cross-reference with 6.2B signal ===
SIGNAL_62B = 'TEST.TEST.Signal.MISO.SPICE_F0P_V6.2B.R1'
CROSS_REF_MONTHS = [pd.Timestamp('2024-12'), pd.Timestamp('2025-01'), pd.Timestamp('2025-02')]

cross_ref_results = []
for am in CROSS_REF_MONTHS:
    auc_str = am.strftime('%Y-%m')
    for ct in CLASS_TYPES:
        label = f"{auc_str}/f0/{ct}"
        try:
            sig_62b = ConstraintsSignal('miso', SIGNAL_62B, 'f0', ct).load_data(am)
            sig_67b = ConstraintsSignal('miso', SIGNAL_NAME, 'f0', ct).load_data(am)
        except Exception as e:
            print(f"  SKIP {label}: {e}")
            continue

        ids_62b = set(sig_62b['constraint_id'])
        ids_67b = set(sig_67b['constraint_id'])
        overlap = ids_67b & ids_62b
        overlap_pct = len(overlap) / max(len(ids_67b), 1) * 100

        tier_corr = float('nan')
        if overlap:
            m = sig_67b[['constraint_id', 'tier']].merge(
                sig_62b[['constraint_id', 'tier']], on='constraint_id',
                suffixes=('_67b', '_62b'),
            )
            if len(m) > 1:
                tier_corr = m['tier_67b'].corr(m['tier_62b'])

        sp_mean_67b = sig_67b['predicted_shadow_price'].mean()
        sp_mean_62b = sig_62b['predicted_shadow_price'].mean()

        cross_ref_results.append({
            'auction_month': auc_str, 'class_type': ct,
            'n_67b': len(sig_67b), 'n_62b': len(sig_62b),
            'overlap': len(overlap), 'overlap_pct': f"{overlap_pct:.0f}%",
            'tier_corr': f"{tier_corr:.3f}",
            'sp_mean_67b': f"{sp_mean_67b:.1f}",
            'sp_mean_62b': f"{sp_mean_62b:.1f}",
        })
        print(f"  {label}: 6.7B={len(sig_67b)}, 6.2B={len(sig_62b)}, overlap={len(overlap)} ({overlap_pct:.0f}%), tier_corr={tier_corr:.3f}")

if cross_ref_results:
    print("\nCross-reference summary:")
    display(pd.DataFrame(cross_ref_results))

In [ ]:
# === evaluate_signal_v2 for f0/onpeak and f0/offpeak ===
aptools = MisoApTools()

eval_results = {}
for class_type in CLASS_TYPES:
    label = f"f0/{class_type}"
    try:
        metrics_df, for_recall, for_precision, signal_dict, _ = aptools.tools.evaluate_signal_v2(
            signal_name=SIGNAL_NAME,
            period_type='f0',
            peak_type=class_type,
            auction_month_st='2024-06',
            auction_month_et_in='2025-12',
            n_jobs=12,
        )
        eval_results[label] = metrics_df
        print(f"\n{label}:")
        display(metrics_df[['auction_month', 'sp', 'sp%,t~0', 'recall,t~0', 'prec,t~0', 'F1,t~0']])
    except Exception as e:
        print(f"\n{label}: FAILED - {e}")
        eval_results[label] = None

print(f"\nMemory after evaluation: {mem_mb():.0f} MB")

In [ ]:
# === Acceptance criteria check ===
# recall,t~0 > 0.30, prec,t~0 > 0.15, sp%,t~0 > 0.40
acceptance_failures = []

for label, mdf in eval_results.items():
    if mdf is None:
        acceptance_failures.append((label, "evaluation failed"))
        continue

    avg_recall = mdf['recall,t~0'].mean()
    avg_prec = mdf['prec,t~0'].mean()
    avg_sp_pct = mdf['sp%,t~0'].mean()

    print(f"{label}: recall={avg_recall:.3f}, prec={avg_prec:.3f}, sp%={avg_sp_pct:.3f}")

    if avg_recall < 0.30:
        acceptance_failures.append((label, f"recall,t~0 = {avg_recall:.3f} < 0.30"))
    if avg_prec < 0.15:
        acceptance_failures.append((label, f"prec,t~0 = {avg_prec:.3f} < 0.15"))
    if avg_sp_pct < 0.40:
        acceptance_failures.append((label, f"sp%,t~0 = {avg_sp_pct:.3f} < 0.40"))

if acceptance_failures:
    print(f"\nWARNING: {len(acceptance_failures)} acceptance criteria not met:")
    for label, msg in acceptance_failures:
        print(f"  {label}: {msg}")
else:
    print("\nAll acceptance criteria PASSED.")

In [ ]:
# === Cleanup ===
import ray
ray.shutdown()
gc.collect()
print(f"Ray shutdown. Final memory: {mem_mb():.0f} MB")

# Summary
log_df = pd.DataFrame(results_log)
n_saved = len(log_df[log_df['status'] == 'saved'])
n_errors = len(log_df[log_df['status'].str.startswith('error')])
n_no_results = len(log_df[log_df['status'] == 'no_results'])
print(f"\nFinal summary: {n_saved} saved, {n_errors} errors, {n_no_results} no results")
print(f"Expected: {len(AUCTION_MONTHS) * len(CLASS_TYPES)} total")